In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd

import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMClassifier
import shap

from src.data_prepatation import create_batched_splits

In [ ]:
from config import dir_config
compiled_dir = dir_config.data.compiled
processed_dir = dir_config.data.processed

In [ ]:
def lgb_cuda_available():
    try:
        import lightgbm as lgb
        data = lgb.Dataset([[1,2],[3,4]], label=[0,1])
        lgb.train({"device": "cuda", "verbose": -1}, data, num_boost_round=1)
        return True
    except Exception:
        return False

print(lgb_cuda_available())

In [ ]:

sleep_stage_cols = ['sleep_stage', 'sleep_stage_transition', 'sleep_stage_trans_prop']
apnea_cols = ['apnea_obstructive', 'apnea_central', 'apnea_hypopnea', 'apnea_mixed']
epoch_cols = ['epoch_id', 'epoch_start', 'epoch_end']
subject_col = []
additional_cols = ['ahi', 'oahi', 'arousal_index']

feature_max_lag = 5
target_type = 'apnea'
target_future_step = 1


USE_GPU = lgb_cuda_available()

# Load data

In [ ]:
# get all parquet files in the processed directory
parquet_files = list(Path(processed_dir).glob('*_with_metadata.parquet'))

# remove "agg_data_with_metadata.parquet" from the list of parquet files
parquet_files = [f for f in parquet_files if f.name != "agg_data_with_metadata.parquet"]

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report, confusion_matrix

def evaluate_model(model, X, y, threshold=0.5):
    y_pred_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_pred_proba >= threshold).astype(int)
    auc_roc = roc_auc_score(y, y_pred_proba)
    auc_pr = average_precision_score(y, y_pred_proba)
    f1 = f1_score(y, y_pred)
    #
    classification = classification_report(y, y_pred)
    confusion = confusion_matrix(y, y_pred)
    return {'auc_roc': auc_roc, 'auc_pr': auc_pr, 'f1': f1, 'classification_report': classification, 'confusion_matrix': confusion}

## Chunk in 0.5hr

In [ ]:
train_X, train_y, val_X, val_y, test_X, test_y, left_out = create_batched_splits(
    parquet_files,
    batch_size=360,
    gap_size=6,
    top_features=None,
    top_features_lag=0,
    target_type='apnea',
    target_future_steps=target_future_step,
    val_ratio=0.2,
    test_ratio=0.2,
    n_leave_out=5,
    random_seed=2542,
)

### XGBoost

In [ ]:
# ## xgboost model
# # for binary classification
# neg = (train_y == 0).sum()
# pos = (train_y == 1).sum()

# model_xgb = xgb.XGBClassifier(
#     n_estimators=100,
#     max_depth=5,
#     scale_pos_weight=neg/pos,  # adjust for class imbalance
#     early_stopping_rounds=10,
#     learning_rate=0.1,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=2542,
#     eval_metric='aucpr'
# )

# model_xgb.fit(train_X, train_y, eval_set=[(val_X, val_y)], verbose=True)


In [ ]:
# # Get model evaluation metrics for the train set
# print("XGBoost Model Evaluation on Train Set:")
# metrics = evaluate_model(model_xgb, train_X, train_y)
# print(f"AUC-ROC: {metrics['auc_roc']:.4f}")
# print(f"AUC-PR: {metrics['auc_pr']:.4f}")
# print(f"F1 Score: {metrics['f1']:.4f}")
# print("Classification Report:")
# print(metrics['classification_report'])

# # Get model evaluation metrics for the validation set
# print("XGBoost Model Evaluation on Validation Set:")
# metrics = evaluate_model(model_xgb, val_X, val_y)
# print(f"AUC-ROC: {metrics['auc_roc']:.4f}")
# print(f"AUC-PR: {metrics['auc_pr']:.4f}")
# print(f"F1 Score: {metrics['f1']:.4f}")
# print("Classification Report:")
# print(metrics['classification_report'])

In [ ]:
# from xgboost import plot_importance
# import pandas as pd

# # Get top 20 most important features
# importance = pd.Series(model_xgb.feature_importances_, index=train_X.columns)
# top_features = importance.nlargest(20).index.tolist()

# print(top_features)

In [ ]:
# import shap

# # Set base score explicitly to avoid the parsing issue
# model_xgb.get_booster().set_param({'base_score': 0.5})

# explainer = shap.TreeExplainer(model_xgb.get_booster())
# shap_values = explainer.shap_values(val_X)

# shap.summary_plot(shap_values, val_X, max_display=20)

### LightGBM

In [ ]:
import optuna
from sklearn.metrics import average_precision_score

optuna.logging.set_verbosity(optuna.logging.WARNING)


neg = (train_y == 0).sum()
pos = (train_y == 1).sum()
spw = neg / pos

BASE_PARAMS = {
    "n_estimators": 5000,
    "scale_pos_weight": spw,
    "random_state": 42,
    "metric": "average_precision",
    "verbosity": -1,
    "n_jobs": 1,                          # avoid thread contention during parallel trials
    "device": "cuda" if USE_GPU else "cpu",
}

FIT_CALLBACKS = [
    lgb.early_stopping(50, first_metric_only=True, verbose=False),
    lgb.log_evaluation(-1),
]


def objective(trial):
    params = BASE_PARAMS | {
        "learning_rate":      trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "max_depth":          trial.suggest_int("max_depth", 3, 7),
        "num_leaves":         trial.suggest_int("num_leaves", 15, 63),
        "min_child_samples":  trial.suggest_int("min_child_samples", 20, 100),
        "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.3, 0.8),
        "subsample":          trial.suggest_float("subsample", 0.4, 0.9),
        "reg_alpha":          trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda":         trial.suggest_float("reg_lambda", 0.0, 4.0),
    }

    model = LGBMClassifier(**params)
    model.fit(
        train_X, train_y,
        eval_set=[(val_X, val_y)],
        callbacks=FIT_CALLBACKS,
    )

    val_proba = model.predict_proba(val_X)[:, 1]
    return average_precision_score(val_y, val_proba)


study = optuna.create_study(direction="maximize")
study.optimize(
    objective,
    n_trials=50,
    n_jobs=-1,              # ← parallel trials across CPU cores
    show_progress_bar=True,
)

print(f"\nBest val AUC-PR: {study.best_value:.4f}")
print(f"Best params:     {study.best_params}")

# --- Retrain best model, use val only for early stopping ---
best_model_params = BASE_PARAMS | study.best_params

model_lgb_tuned = LGBMClassifier(**best_model_params)
model_lgb_tuned.fit(
    train_X, train_y,
    eval_set=[(val_X, val_y)],
    callbacks=FIT_CALLBACKS,
)

print("\n--- Tuned model evaluation ---")
for split_name, X, y in [
    ("Train", train_X, train_y),
    ("Val",   val_X,   val_y),
]:
    m = evaluate_model(model_lgb_tuned, X, y)
    print(f"{split_name:5s}  AUC-ROC={m['auc_roc']:.4f}  AUC-PR={m['auc_pr']:.4f}  F1={m['f1']:.4f}")

In [ ]:
explainer = shap.TreeExplainer(model_lgb_tuned)

# --- Validation set SHAP: for plotting/evaluation ---
val_shap_values = explainer.shap_values(val_X)
val_sv = val_shap_values[1] if isinstance(val_shap_values, list) else val_shap_values
shap.summary_plot(val_sv, val_X, max_display=20)

# --- Train set SHAP: for feature selection ---
# Using train set here to select features for the lagged model;
# val/test sets are kept unseen for downstream evaluation.
train_shap_values = explainer.shap_values(train_X)
train_sv = train_shap_values[1] if isinstance(train_shap_values, list) else train_shap_values

shap_importance = pd.Series(
    np.abs(train_sv).mean(axis=0),
    index=train_X.columns,
).sort_values(ascending=False)

# Filter out subject metadata columns from feature candidates
subject_metadata = pd.read_csv(Path(processed_dir) / 'participant_info.csv')
metadata_cols = set(subject_metadata.columns)

top_shap_features = [
    col for col in shap_importance.nlargest(20).index
    if col not in metadata_cols
]

dropped = [col for col in shap_importance.nlargest(20).index if col in metadata_cols]
if dropped:
    print(f"Dropped metadata columns from top features: {dropped}")

print(f"Top SHAP features: {top_shap_features}")

In [ ]:
# Let's look at pairwise correlations among the top SHAP features to check for multicollinearity
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 10))
sns.heatmap(train_X[top_shap_features].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix of Top SHAP Features (Train Set)")
plt.show()

In [ ]:
# let's get all pairs of features with correlation > 0.8
corr_matrix = train_X[top_shap_features].corr().abs()
high_corr_pairs = np.where(corr_matrix > 0.8)
high_corr_pairs = [(train_X[top_shap_features].columns[i], train_X[top_shap_features].columns[j], corr_matrix.iloc[i, j])
                    for i, j in zip(*high_corr_pairs) if i < j]

high_corr_pairs

In [ ]:
features_to_drop = ["resp_flow_max", "resp_flow_range", "sao2_range", "resp_ptaf_max", ]

### Adding lag features for top SHAP features

In [ ]:
lagged5_train_X, future0_train_y, lagged5_val_X, future0_val_y, lagged5_test_X, future0_test_y, left_out = create_batched_splits(
    parquet_files,
    batch_size=360,
    gap_size=6,
    top_features=top_shap_features,
    top_features_lag=feature_max_lag,
    target_type='apnea',
    target_future_steps=target_future_step,
    val_ratio=0.2,
    test_ratio=0.2,
    n_leave_out=5,
    random_seed=2542,
)

======

In [ ]:
# give me frequency of these features in highly correlated pairs
from collections import Counter
feature_freq = Counter()
for f1, f2, corr in high_corr_pairs:
    feature_freq[f1] += 1
    feature_freq[f2] += 1

feature_freq

In [ ]:
# if a feature appears more than once, give me the rows with all three (for twice) or all four (for thrice) features so I can decide which one to drop
for feature, freq in feature_freq.items():
    if freq > 1:
        correlated_features = [f2 for f1, f2, corr in high_corr_pairs if f1 == feature] + [f1 for f1, f2, corr in high_corr_pairs if f2 == feature]
        correlated_features.append(feature)
        print(f"\nFeature '{feature}' appears in {freq} highly correlated pairs. Correlated features: {correlated_features}")
        display(lagged5_train_X[correlated_features].head())


In [ ]:
# Let's visualize it better so we can decide which features to drop from the lagged model
for f1, f2, corr in high_corr_pairs:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=lagged5_train_X[f1], y=lagged5_train_X[f2], alpha=0.5)
    plt.title(f"Scatter plot of {f1} vs {f2} (corr={corr:.2f})")
    plt.xlabel(f1)
    plt.ylabel(f2)
    plt.show()

======

In [ ]:
# neg = (future0_train_y == 0).sum()
# pos = (future0_train_y == 1).sum()

# model_lgb_lagged5_future0 = LGBMClassifier(
#     n_estimators=5000,
#     max_depth=4,
#     learning_rate=0.02,
#     scale_pos_weight=neg/pos,
#     colsample_bytree=0.4,     # reduced from 0.6 — more features per tree increases collinearity noise
#     subsample=0.6,
#     min_child_samples=50,
#     reg_alpha=0.5,            # increased from 0.2 — stronger L1 to prune redundant lag features
#     reg_lambda=2.0,           # increased from 1.0 — stronger L2 for expanded feature space
#     num_leaves=31,
#     random_state=42,
#     metric='average_precision'
# )

# model_lgb_lagged5_future0.fit(
#     lagged5_train_X, future0_train_y,
#     eval_set=[(lagged5_val_X, future0_val_y)],
#     callbacks=[
#         lgb.early_stopping(50, first_metric_only=True),
#         lgb.log_evaluation(25)
#     ],
# )


In [ ]:
# # Get model evaluation metrics for the train set
# print("LightGBM Model Evaluation on Train Set:")
# metrics = evaluate_model(model_lgb_lagged5_future0, lagged5_train_X, future0_train_y)
# print(f"AUC-ROC: {metrics['auc_roc']:.4f}")
# print(f"AUC-PR: {metrics['auc_pr']:.4f}")
# print(f"F1 Score: {metrics['f1']:.4f}")
# print("Classification Report:")
# print(metrics['classification_report'])

# # Get model evaluation metrics for the validation set
# print("LightGBM Model Evaluation on Validation Set:")
# metrics = evaluate_model(model_lgb_lagged5_future0, lagged5_val_X, future0_val_y)
# print(f"AUC-ROC: {metrics['auc_roc']:.4f}")
# print(f"AUC-PR: {metrics['auc_pr']:.4f}")
# print(f"F1 Score: {metrics['f1']:.4f}")
# print("Classification Report:")
# print(metrics['classification_report'])

In [ ]:
# explainer = shap.TreeExplainer(model_lgb_lagged5_future0)
# shap_values = explainer.shap_values(lagged5_val_X)

# shap.summary_plot(shap_values, lagged5_val_X, max_display=20)

Hyperparameter tuning

In [ ]:
import optuna
from sklearn.metrics import average_precision_score

optuna.logging.set_verbosity(optuna.logging.WARNING)


neg = (future0_train_y == 0).sum()
pos = (future0_train_y == 1).sum()
spw = neg / pos

BASE_PARAMS = {
    "n_estimators": 5000,
    "scale_pos_weight": spw,
    "random_state": 42,
    "metric": "average_precision",
    "verbosity": -1,
    "n_jobs": 1,                          # avoid thread contention during parallel trials
    **({"device": "gpu"} if USE_GPU else {}),
}

FIT_CALLBACKS = [
    lgb.early_stopping(50, first_metric_only=True, verbose=False),
    lgb.log_evaluation(-1),
]


def objective(trial):
    params = BASE_PARAMS | {
        "learning_rate":      trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "max_depth":          trial.suggest_int("max_depth", 3, 7),
        "num_leaves":         trial.suggest_int("num_leaves", 15, 63),
        "min_child_samples":  trial.suggest_int("min_child_samples", 20, 100),
        "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.3, 0.8),
        "subsample":          trial.suggest_float("subsample", 0.4, 0.9),
        "reg_alpha":          trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda":         trial.suggest_float("reg_lambda", 0.0, 4.0),
    }

    model = LGBMClassifier(**params)
    model.fit(
        lagged5_train_X, future0_train_y,
        eval_set=[(lagged5_val_X, future0_val_y)],
        callbacks=FIT_CALLBACKS,
    )

    val_proba = model.predict_proba(lagged5_val_X)[:, 1]
    return average_precision_score(future0_val_y, val_proba)


study = optuna.create_study(direction="maximize")
study.optimize(
    objective,
    n_trials=50,
    n_jobs=-1,              # ← parallel trials across CPU cores
    show_progress_bar=True,
)

print(f"\nBest val AUC-PR: {study.best_value:.4f}")
print(f"Best params:     {study.best_params}")

# --- Retrain best model, use val only for early stopping ---
best_model_params = BASE_PARAMS | study.best_params

model_lgb_lagged5_future0_tuned = LGBMClassifier(**best_model_params)
model_lgb_lagged5_future0_tuned.fit(
    lagged5_train_X, future0_train_y,
    eval_set=[(lagged5_val_X, future0_val_y)],
    callbacks=FIT_CALLBACKS,
)

print("\n--- Tuned model evaluation ---")
for split_name, X, y in [
    ("Train", lagged5_train_X, future0_train_y),
    ("Val",   lagged5_val_X,   future0_val_y),
]:
    m = evaluate_model(model_lgb_lagged5_future0_tuned, X, y)
    print(f"{split_name:5s}  AUC-ROC={m['auc_roc']:.4f}  AUC-PR={m['auc_pr']:.4f}  F1={m['f1']:.4f}")

### With Only Top shap features

In [ ]:
# train new model with only top 20 SHAP features (no metadata) and their lags up to 5 steps, then evaluate on val set
explainer = shap.TreeExplainer(model_lgb_lagged5_future0_tuned)

# --- Validation set SHAP: for plotting/evaluation ---
val_shap_values = explainer.shap_values(lagged5_val_X)
val_sv = val_shap_values[1] if isinstance(val_shap_values, list) else val_shap_values
shap.summary_plot(val_sv, lagged5_val_X, max_display=20)

# --- Train set SHAP: for feature selection ---
# Using train set here to select features for the lagged model;
# val/test sets are kept unseen for downstream evaluation.
train_shap_values = explainer.shap_values(lagged5_train_X)
train_sv = train_shap_values[1] if isinstance(train_shap_values, list) else train_shap_values

shap_importance = pd.Series(
    np.abs(train_sv).mean(axis=0),
    index=lagged5_train_X.columns,
).sort_values(ascending=False)

# Filter out subject metadata columns from feature candidates
subject_metadata = pd.read_csv(Path(processed_dir) / 'participant_info.csv')
metadata_cols = set(subject_metadata.columns)

top_shap_features = [
    col for col in shap_importance.nlargest(20).index
    if col not in metadata_cols
]

dropped = [col for col in shap_importance.nlargest(20).index if col in metadata_cols]
if dropped:
    print(f"Dropped metadata columns from top features: {dropped}")

print(f"Top SHAP features: {top_shap_features}")


In [ ]:
# remove all columns except the top 20 SHAP features and their lags up to 5 steps from lagged5_shape20_train_X and lagged5_shape20_val_X
lagged5_shap20_train_X = lagged5_train_X[top_shap_features]
lagged5_shap20_val_X = lagged5_val_X[top_shap_features]

In [ ]:
import optuna
from sklearn.metrics import average_precision_score

optuna.logging.set_verbosity(optuna.logging.WARNING)


neg = (future0_train_y == 0).sum()
pos = (future0_train_y == 1).sum()
spw = neg / pos

BASE_PARAMS = {
    "n_estimators": 5000,
    "scale_pos_weight": spw,
    "random_state": 42,
    "metric": "average_precision",
    "verbosity": -1,
    "n_jobs": 1,                          # avoid thread contention during parallel trials
    **({"device": "gpu"} if USE_GPU else {}),
}

FIT_CALLBACKS = [
    lgb.early_stopping(50, first_metric_only=True, verbose=False),
    lgb.log_evaluation(-1),
]


def objective(trial):
    params = BASE_PARAMS | {
        "learning_rate":      trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "max_depth":          trial.suggest_int("max_depth", 3, 7),
        "num_leaves":         trial.suggest_int("num_leaves", 15, 63),
        "min_child_samples":  trial.suggest_int("min_child_samples", 20, 100),
        "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.3, 0.8),
        "subsample":          trial.suggest_float("subsample", 0.4, 0.9),
        "reg_alpha":          trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda":         trial.suggest_float("reg_lambda", 0.0, 4.0),
    }

    model = LGBMClassifier(**params)
    model.fit(
        lagged5_shap20_train_X, future0_train_y,
        eval_set=[(lagged5_shap20_val_X, future0_val_y)],
        callbacks=FIT_CALLBACKS,
    )

    val_proba = model.predict_proba(lagged5_shap20_val_X)[:, 1]
    return average_precision_score(future0_val_y, val_proba)


study = optuna.create_study(direction="maximize")
study.optimize(
    objective,
    n_trials=50,
    n_jobs=-1,              # ← parallel trials across CPU cores
    show_progress_bar=True,
)

print(f"\nBest val AUC-PR: {study.best_value:.4f}")
print(f"Best params:     {study.best_params}")

# --- Retrain best model, use val only for early stopping ---
best_model_params = BASE_PARAMS | study.best_params

model_lgb_lagged5_shap20_future0_tuned = LGBMClassifier(**best_model_params)
model_lgb_lagged5_shap20_future0_tuned.fit(
    lagged5_shap20_train_X, future0_train_y,
    eval_set=[(lagged5_shap20_val_X, future0_val_y)],
    callbacks=FIT_CALLBACKS,
)

print("\n--- Tuned model evaluation ---")
for split_name, X, y in [
    ("Train", lagged5_shap20_train_X, future0_train_y),
    ("Val",   lagged5_shap20_val_X,   future0_val_y),
]:
    m = evaluate_model(model_lgb_lagged5_shap20_future0_tuned, X, y)
    print(f"{split_name:5s}  AUC-ROC={m['auc_roc']:.4f}  AUC-PR={m['auc_pr']:.4f}  F1={m['f1']:.4f}")